# LLM Few-Shot Sentiment Analysis — Local Debug Notebook

**Goal:** Use Mistral-7B-Instruct with 5-shot prompting to classify guitar reviews as `positive` / `negative` / `N/A`.

**This notebook:** Local debug mode — runs 50 samples on your Mac using **Ollama** (fast, Apple-Silicon-optimized).

---
### One-time setup (run in Terminal before opening this notebook)
```bash
# 1. Install Ollama (if not already installed)
brew install ollama

# 2. Start the Ollama server (keep this terminal open)
ollama serve

# 3. In a NEW terminal tab — pull the Mistral model (~4 GB, 4-bit quantized)
ollama pull mistral

# 4. Install Python dependencies (once)
pip install requests pandas pyarrow scikit-learn tqdm
```
After setup, run all cells top to bottom.

In [1]:
import json
import re
import time
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm

print('All imports OK')

All imports OK


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Toggle: True = 50-sample local debug | False = full dataset (HPC only)
DEBUG_MODE = True
DEBUG_SAMPLES = 50

# Fixed project parameters — DO NOT CHANGE
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# Ollama settings (local Mac)
OLLAMA_URL   = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'mistral'       # matches what you pulled: ollama pull mistral

# Path to the dataset
DATA_PATH = '../../guitars_with_topics_v2.parquet'

# Output files
OUT_PREDICTIONS   = 'llm_predictions_debug.csv'
OUT_TOPIC_SUMMARY = 'llm_topic_sentiment_debug.csv'

print(f'DEBUG_MODE = {DEBUG_MODE}')
print(f'Model      = {OLLAMA_MODEL} via Ollama')

DEBUG_MODE = True
Model      = mistral via Ollama


In [3]:
# ── Verify Ollama is running ───────────────────────────────────────────────────

def check_ollama():
    """Ping Ollama server and confirm the model is available."""
    try:
        resp = requests.get('http://localhost:11434/api/tags', timeout=5)
        if resp.status_code != 200:
            raise ConnectionError('Ollama responded with non-200 status.')
        models = [m['name'] for m in resp.json().get('models', [])]
        print('Ollama is running. Available models:', models)
        if not any(OLLAMA_MODEL in m for m in models):
            print(f"WARNING: '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}")
        else:
            print(f"Model '{OLLAMA_MODEL}' is ready.")
    except Exception as e:
        print('ERROR: Cannot connect to Ollama.')
        print('  Make sure you ran: ollama serve  (in a separate terminal)')
        raise e

check_ollama()

Ollama is running. Available models: ['mistral:latest']
Model 'mistral' is ready.


In [4]:
# ── Load and prepare data ──────────────────────────────────────────────────────

df_raw = pd.read_parquet(DATA_PATH)
print(f'Raw dataset: {len(df_raw):,} rows, columns: {list(df_raw.columns)}')

# Detect the rating column name (handles both 'overall' and 'rating')
rating_col = 'overall' if 'overall' in df_raw.columns else 'rating'

# Build sentiment labels from ratings (same rule as BERT)
# Rating 4-5 → positive | Rating 1-2 → negative | Rating 3 → dropped
df = df_raw[df_raw[rating_col] != 3].copy()
df['sentiment'] = df[rating_col].apply(lambda r: 'positive' if r >= 4 else 'negative')

# Detect the review text column name (handles 'reviewText', 'review_text', 'text')
for candidate in ['reviewText', 'review_text', 'text']:
    if candidate in df.columns:
        text_col = candidate
        break
print(f'Using columns: rating="{rating_col}", text="{text_col}"')

# Drop rows with missing review text
df = df.dropna(subset=[text_col]).reset_index(drop=True)
df['text'] = df[text_col].astype(str).str.strip()

print(f'After cleaning: {len(df):,} reviews')
print(df['sentiment'].value_counts())

Raw dataset: 134,068 rows, columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'subcategory', 'store', 'average_rating', 'price', 'topic_id', 'topic_label', 'topic_prob']
Using columns: rating="rating", text="text"
After cleaning: 124,243 reviews
sentiment
positive    105720
negative     18523
Name: count, dtype: int64


In [5]:
# ── Train/test split (same seed as BERT for fair comparison) ───────────────────

train_df, test_df = train_test_split(
    df,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = df['sentiment']
)

print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

# In debug mode, take only a small slice of the test set
if DEBUG_MODE:
    eval_df = test_df.sample(n=DEBUG_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'DEBUG: evaluating on {len(eval_df)} samples')
else:
    eval_df = test_df.reset_index(drop=True)
    print(f'FULL: evaluating on {len(eval_df)} samples')

print(eval_df['sentiment'].value_counts())

Train: 99,394  |  Test: 24,849
DEBUG: evaluating on 50 samples
sentiment
positive    44
negative     6
Name: count, dtype: int64


In [6]:
# ── Build 5-shot examples from the TRAINING set ────────────────────────────────
# We pick 5 examples (3 positive + 2 negative) to cover both classes.
# These are fixed so every prompt is identical — important for reproducibility.

np.random.seed(RANDOM_STATE)

# Sample a few candidates then hand-pick the clearest ones
pos_candidates = train_df[train_df['sentiment'] == 'positive'].sample(20, random_state=RANDOM_STATE)
neg_candidates = train_df[train_df['sentiment'] == 'negative'].sample(20, random_state=RANDOM_STATE)

# Print candidates so you can inspect and choose the best 5
print('=== Positive candidates (pick 3) ===')
for i, row in pos_candidates.head(5).iterrows():
    print(f"[{i}] {row['text'][:120]}")
    print()

print('=== Negative candidates (pick 2) ===')
for i, row in neg_candidates.head(5).iterrows():
    print(f"[{i}] {row['text'][:120]}")
    print()

=== Positive candidates (pick 3) ===
[93697] Everything you need for a child to get started

[34219] Perfect gift for the budding guitar player. My nephew loved it and didn't put it down for hours. It looks way more expen

[4328] As many of these reviews have stated, not only is this a good &#34;travel guitar&#34;, but it's just a pretty good 'guit

[94720] When I received this order, the guitar also came with a case to keep the guitar in. I was really amazed by that because 

[34991] Makes tuning incredibly easy. Simply turn it on and tune your guitar to the middle green bar & voila! Lifesaver for me s

=== Negative candidates (pick 2) ===
[92221] I was really looking forward to getting my new guitar and use it over the Thanksgiving break and was super excited. Howe

[52135] I wanted to love this guitar! I have played for many years, I bought this to teach my daughters how to play, as my squar

[74356] Today I received the purple burst version of the guitar. I do like the looks of it 

In [7]:
# ── Fixed 5-shot examples (loaded from training set) ──────────────────────────
# Positive indices: [34991], [34219], [4328]
# Negative indices: [92221], [52135]

pos_indices = [34991, 34219, 4328]
neg_indices = [92221, 52135]

FEW_SHOT_EXAMPLES = []
for idx in pos_indices:
    FEW_SHOT_EXAMPLES.append({'text': train_df.loc[idx, 'text'], 'label': 'positive'})
for idx in neg_indices:
    FEW_SHOT_EXAMPLES.append({'text': train_df.loc[idx, 'text'], 'label': 'negative'})

print(f'{len(FEW_SHOT_EXAMPLES)} few-shot examples loaded from training set.')
for ex in FEW_SHOT_EXAMPLES:
    print(f"  [{ex['label']}] {ex['text'][:120]}...")

5 few-shot examples loaded from training set.
  [positive] Makes tuning incredibly easy. Simply turn it on and tune your guitar to the middle green bar & voila! Lifesaver for me s...
  [positive] Perfect gift for the budding guitar player. My nephew loved it and didn't put it down for hours. It looks way more expen...
  [positive] As many of these reviews have stated, not only is this a good &#34;travel guitar&#34;, but it's just a pretty good 'guit...
  [negative] I was really looking forward to getting my new guitar and use it over the Thanksgiving break and was super excited. Howe...
  [negative] I wanted to love this guitar! I have played for many years, I bought this to teach my daughters how to play, as my squar...


In [8]:
# ── Prompt builder ─────────────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    'You are a sentiment classifier for guitar product reviews. '
    'Your task is to classify each review as either "positive" or "negative". '
    'You MUST respond with exactly one word — either positive or negative. '
    'If the review is mostly positive, respond: positive. '
    'If the review is mostly negative, respond: negative. '
    'Never respond with anything else. No explanation. No punctuation. Just one word.'
)

def build_prompt(review_text: str) -> str:
    """Build a 5-shot prompt for a single review."""
    lines = [SYSTEM_PROMPT, '']

    # Add few-shot examples
    for i, ex in enumerate(FEW_SHOT_EXAMPLES, 1):
        # Truncate each example to 300 chars to keep prompt short
        ex_text = ex['text'][:300] + ('...' if len(ex['text']) > 300 else '')
        lines.append(f'Review: {ex_text}')
        lines.append(f'Sentiment: {ex["label"]}')
        lines.append('')

    # Add the target review (truncate to 400 chars)
    truncated = review_text[:400] + ('...' if len(review_text) > 400 else '')
    lines.append(f'Review: {truncated}')
    lines.append('Sentiment:')

    return '\n'.join(lines)

# Quick sanity check
sample_prompt = build_prompt('This guitar sounds amazing, best I have ever owned!')
print(sample_prompt)
print(f'\nPrompt length: {len(sample_prompt)} chars')

You are a sentiment classifier for guitar product reviews. Your task is to classify each review as either "positive" or "negative". You MUST respond with exactly one word — either positive or negative. If the review is mostly positive, respond: positive. If the review is mostly negative, respond: negative. Never respond with anything else. No explanation. No punctuation. Just one word.

Review: Makes tuning incredibly easy. Simply turn it on and tune your guitar to the middle green bar & voila! Lifesaver for me since I'm terrible at tuning my guitar.
Sentiment: positive

Review: Perfect gift for the budding guitar player. My nephew loved it and didn't put it down for hours. It looks way more expensive than it  cost and has lots of features.
Sentiment: positive

Review: As many of these reviews have stated, not only is this a good &#34;travel guitar&#34;, but it's just a pretty good 'guitar' too.<br />I also needed to adjust the action, and I swapped the stock EMG (HZ or Select series) 

In [9]:
# ── Ollama inference function ──────────────────────────────────────────────────

def parse_label(raw_output: str) -> str:
    """Extract positive / negative / N/A from the model's raw text."""
    text = raw_output.strip().lower()
    # Remove punctuation and take first word
    first_word = re.split(r'[^a-z/]', text)[0]
    if first_word in ('positive', 'pos'):
        return 'positive'
    if first_word in ('negative', 'neg'):
        return 'negative'
    # Fallback: search anywhere in the output
    if 'positive' in text:
        return 'positive'
    if 'negative' in text:
        return 'negative'
    return 'N/A'


def query_ollama(prompt: str, timeout: int = 120) -> tuple[str, str]:
    """
    Send a prompt to the local Ollama server.
    Returns (raw_output, parsed_label).
    """
    payload = {
        'model':  OLLAMA_MODEL,
        'prompt': prompt,
        'stream': False,
        'options': {
            'temperature': 0.0,   # greedy decoding for reproducibility
            'num_predict': 10,    # we only need one word
            'top_p': 1.0,
        }
    }
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
        resp.raise_for_status()
        raw = resp.json().get('response', '').strip()
        return raw, parse_label(raw)
    except requests.exceptions.Timeout:
        return 'TIMEOUT', 'N/A'
    except Exception as e:
        return f'ERROR: {e}', 'N/A'


# Quick test with one review
test_prompt = build_prompt('This guitar is amazing, best I have ever owned!')
raw_out, label = query_ollama(test_prompt)
print(f'Raw output : "{raw_out}"')
print(f'Parsed label: {label}')

Raw output : "positive"
Parsed label: positive


In [10]:
# ── Run inference on eval set ──────────────────────────────────────────────────

print(f'Running inference on {len(eval_df)} reviews...')
print('(Each review takes ~1-3 seconds with Ollama on Apple Silicon)')
print()

raw_outputs = []
pred_labels = []
start_time  = time.time()

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc='Classifying'):
    prompt = build_prompt(row['text'])
    raw, label = query_ollama(prompt)
    raw_outputs.append(raw)
    pred_labels.append(label)

elapsed = time.time() - start_time
print(f'\nDone in {elapsed:.1f}s  ({elapsed/len(eval_df):.1f}s per review)')

# Attach predictions to dataframe
eval_df = eval_df.copy()
eval_df['raw_output']      = raw_outputs
eval_df['pred_sentiment']  = pred_labels

# Show prediction distribution
print('\nPrediction distribution:')
print(eval_df['pred_sentiment'].value_counts())

Running inference on 50 reviews...
(Each review takes ~1-3 seconds with Ollama on Apple Silicon)



Classifying: 100%|██████████| 50/50 [01:02<00:00,  1.25s/it]



Done in 62.4s  (1.2s per review)

Prediction distribution:
pred_sentiment
positive    41
negative     8
N/A          1
Name: count, dtype: int64


In [11]:
# ── Evaluate performance ───────────────────────────────────────────────────────

# Only evaluate on rows where the model gave a valid answer (not N/A)
valid_mask = eval_df['pred_sentiment'] != 'N/A'
valid_df   = eval_df[valid_mask]

na_count = (~valid_mask).sum()
print(f'N/A responses: {na_count}/{len(eval_df)} ({100*na_count/len(eval_df):.1f}%)')
print()

if len(valid_df) > 0:
    y_true = valid_df['sentiment']
    y_pred = valid_df['pred_sentiment']
    
    macro_f1 = f1_score(y_true, y_pred, average='macro', labels=['positive', 'negative'])
    
    print(f'Macro F1 (valid predictions): {macro_f1:.4f}')
    print()
    print(classification_report(y_true, y_pred, labels=['positive', 'negative']))
else:
    print('No valid predictions to evaluate.')

N/A responses: 1/50 (2.0%)

Macro F1 (valid predictions): 0.9167

              precision    recall  f1-score   support

    positive       1.00      0.95      0.98        43
    negative       0.75      1.00      0.86         6

    accuracy                           0.96        49
   macro avg       0.88      0.98      0.92        49
weighted avg       0.97      0.96      0.96        49



In [12]:
# ── Inspect misclassifications ─────────────────────────────────────────────────

wrong = valid_df[valid_df['sentiment'] != valid_df['pred_sentiment']]
print(f'Misclassified: {len(wrong)}/{len(valid_df)} ({100*len(wrong)/len(valid_df):.1f}%)')
print()

for _, row in wrong.head(5).iterrows():
    print(f"True: {row['sentiment']}  |  Pred: {row['pred_sentiment']}  |  Raw: '{row['raw_output']}'")
    print(f"Review: {row['text'][:150]}")
    print()

Misclassified: 2/49 (4.1%)

True: positive  |  Pred: negative  |  Raw: 'negative'
Review: Knocking around. I had no idea how amazingly limp some of these people are. I am in the middle of setting up the neck on a cheap guitar. New to it so 

True: positive  |  Pred: negative  |  Raw: 'negative'
Review: I bought a used one from amazon warehouse, it came almost new. It was however missing the hygrometer, and the wood on the neck was dry. So dry, that t



In [13]:
# ── Topic-level sentiment summary ──────────────────────────────────────────────
# If the dataset has a 'topic' column, compute per-topic sentiment breakdown.

topic_col = None
for candidate in ['topic', 'Topic', 'main_topic', 'assigned_topic']:
    if candidate in eval_df.columns:
        topic_col = candidate
        break

if topic_col:
    # Only use valid predictions for topic summary
    topic_df = valid_df.copy()
    topic_summary = (
        topic_df.groupby(topic_col)['pred_sentiment']
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .rename(columns={'positive': 'pct_positive', 'negative': 'pct_negative'})
        .reset_index()
    )
    topic_summary['total_reviews'] = (
        topic_df.groupby(topic_col).size().values
    )
    # Sort by negative sentiment (most problematic topics first)
    if 'pct_negative' in topic_summary.columns:
        topic_summary = topic_summary.sort_values('pct_negative', ascending=False)
    
    print('Per-topic sentiment (debug sample):')
    print(topic_summary.to_string(index=False))
else:
    print('No topic column found — skipping topic summary.')
    print(f'Available columns: {list(eval_df.columns)}')
    topic_summary = None

No topic column found — skipping topic summary.
Available columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'subcategory', 'store', 'average_rating', 'price', 'topic_id', 'topic_label', 'topic_prob', 'sentiment', 'raw_output', 'pred_sentiment']


In [14]:
# ── Save results ───────────────────────────────────────────────────────────────

# Save per-review predictions
save_cols = ['text', 'sentiment', 'pred_sentiment', 'raw_output']
if topic_col:
    save_cols = [topic_col] + save_cols
eval_df[save_cols].to_csv(OUT_PREDICTIONS, index=False)
print(f'Predictions saved → {OUT_PREDICTIONS}')

# Save topic summary (if available)
if topic_summary is not None:
    topic_summary.to_csv(OUT_TOPIC_SUMMARY, index=False)
    print(f'Topic summary saved → {OUT_TOPIC_SUMMARY}')

print('\nDebug run complete! To run on full dataset, set DEBUG_MODE = False and run on HPC.')

Predictions saved → llm_predictions_debug.csv

Debug run complete! To run on full dataset, set DEBUG_MODE = False and run on HPC.


---
## Next steps

Once this debug run looks correct (predictions make sense, N/A rate is low, F1 is reasonable):

1. **Check the prompts** — if N/A rate is high or F1 is low, tweak `FEW_SHOT_EXAMPLES` or `SYSTEM_PROMPT`
2. **Run on HPC** — use the companion script `llm_fewshot_hpc.py` (transformers pipeline, full 20% test set)
3. **Compare with BERT** — both methods use the same test split, so Macro F1 is directly comparable

### Output files
| File | Description |
|------|-------------|
| `llm_predictions_debug.csv` | Per-review predictions (debug subset) |
| `llm_topic_sentiment_debug.csv` | Per-topic sentiment breakdown (debug) |
| `llm_predictions.csv` | Full results (HPC run) |
| `llm_topic_sentiment.csv` | Full topic summary (HPC run) |